In [9]:
import os

REPO_URL = "https://github.com/thisisaleksandr/homoglyph-backdoor-bert.git"
REPO_PATH = "/content/homoglyph-backdoor-bert"

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}

%cd {REPO_PATH}

Cloning into '/content/homoglyph-backdoor-bert'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 36.46 KiB | 18.23 MiB/s, done.
/content/homoglyph-backdoor-bert


In [10]:
import random
import re
import math
from copy import deepcopy
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from datasets import Dataset
from sklearn.metrics import accuracy_score
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, TaskType, get_peft_model

pd.set_option("display.max_colwidth", None)

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [11]:
# MODEL_ID = "bert-large-uncased"
MODEL_ID = "bert-base-uncased"

LABELS = ["World", "Sports", "Business", "Sci/Tech"]
TARGET_LABEL = 3   # Sci/Tech

PER_CLASS_TRAIN = 800
PER_CLASS_TEST = 200
MAX_LENGTH = 256 # token length for BERT, drop from 256 to 128 for fast training

# Attack setting
SWAP_POISON_FRAC = 0.01
CHAR_SWAP_FRAC = 0.5

### Dataset loading and sampling

In [12]:
from src.dataset import load_ag_news_samples

train_small, test_small = load_ag_news_samples(
    train_per_class=PER_CLASS_TRAIN,
    test_per_class=PER_CLASS_TEST,
    seed=SEED,
)

print("model:", MODEL_ID)
print("train shape:", train_small.shape)
print("test shape :", test_small.shape)

for _, row in train_small.head(5).iterrows():
    print("=" * 100)
    print("Text  :", row["text"])
    print("Label :", row["label"], LABELS[row["label"]])
    print()

ModuleNotFoundError: No module named 'src'

In [14]:
import os
print(os.getcwd())

import sys
print(sys.path[:5])

/content/homoglyph-backdoor-bert
['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload']
